# 08 — Recreate the selection from raw with one config

**You will:** capture the approved synthetic selection as source decisions,
rebuild from raw with `pleb run --config`, compare two independent runs and
reject a changed decision catalog. Keep observed PTA data outside this checkout.

The normal production inputs are raw files, one TOML, the review catalog and
its independent verification reference. No processed checkpoint or old release
is read during a production run. Human decisions are inputs: repeating detection
does not recreate a human review.

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import pandas as pd


COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab  # noqa: E402 -- add the mounted course path first
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

## Declare every stage

Here the kernel already runs in the recorded image, so `timing.native=true`
avoids nested containers. A host launch instead supplies a pinned image and
`timing.native=false`. Backend roots are relative to `source.raw`; other paths
are relative to this TOML. The profile reuses the existing native ingest and
FixDataset operations. Duplicate occurrences are commented, not deleted.

`qc.mode='replay'` uses preserved selection decisions and skips fresh PQC.
`timing.fit=false` demonstrates selection reproduction without refitting. The
generated PARs start from raw models; they are not the fitted models approved
in 06. To request a new fit, set `fit=true`, choose `iterations`, and inspect its
timing logs and coefficients. Selection agreement alone does not validate a model.

In [ ]:
native = CONFIGS/'native'
native.mkdir()
profile = dict(release_version=1, run=dict(mode='release'),
    source=dict(raw='../../raw'),
    ingest=dict(sources=['.'],par_roots=['par'],template_roots=[],pulsars=PULSARS,
        backends={name:dict(root=name,tim_glob='*.tim') for name in ['A','B']}),
    processing=dict(remove_patterns=[],min_toas_per_backend_tim=0,
        dedupe_toas_within_tim=True,dedupe_action='comment',remove_overlaps_exact=False,
        insert_missing_jumps=False,prune_stale_jumps=False),
    clocks=dict(include_container=True,utc_stations=[]),
    qc=dict(mode='replay'),review=dict(mode='catalog',catalog='decisions.json'),
    timing=dict(fit=False,validate=False,native=True,iterations=1,weighted=True,
                ephem='DE440',clock='TT(BIPM2024)'),
    variants=dict(combined=['*']),
    output=dict(directory='../../native-run-1',repository='../../dataset-repo',
                base='release_model',branch='release_recreated',dataset_path='data'),
    verify=dict(reference='expected.json',reject_exact_duplicates=True))
profile_path = lab.write_config(native/'release.toml',profile)
reference = lab.snapshot(REPO,'release_model',ROOT/'exports/08_approved')

## One-time migration of an existing selection

`release-decisions` matches the approved reference to physical raw rows. It fails
on missing measurements or timing-context changes and never copies reference TOA
values into raw. It labels these as **inherited selection** decisions, without
inventing a reviewer or a statistical justification. Distinct duplicate occurrences
have different source IDs. IDs depend on relative raw path, checksum and line.

This import is needed only to migrate an existing release. A new native interactive
review writes its decisions directly using source IDs; no migration is then needed.

In [ ]:
from pleb.release import digest, file_digest

lab.command(CLI+['release-decisions','--config',profile_path,'--reference',reference,
    '--output',native/'decisions.json','--expected',native/'expected.json',
    '--workspace',ROOT/'decision-import'],log=ROOT/'logs/08_import.log')
inventory = json.loads((ROOT/'decision-import/runtime/input_inventory.json').read_text())
profile['source']['inventory_sha256'] = digest(inventory)
profile['review']['sha256'] = file_digest(native/'decisions.json')
profile['verify']['sha256'] = file_digest(native/'expected.json')
lab.write_config(profile_path,profile)
catalog = json.loads((native/'decisions.json').read_text())
display(pd.DataFrame(catalog['decisions']))
assert all(row['origin']=='inherited_selection' for row in catalog['decisions'])

## The production command

The old release is no longer an input. PLEB resolves and checks raw inputs,
ingests them, assembles clocks, normalises flags/duplicates, applies decisions,
builds variants and verifies the selected IDs and measurement contexts. Reports
and runtime files stay beside `dataset/`. Optional Git publication changes only
the requested dataset path on a new local branch; it leaves the checkout alone.

In [ ]:
checkout = lab.command(['git','-C',REPO,'rev-parse','HEAD']).strip()
lab.command(CLI+['run','--config',profile_path],log=ROOT/'logs/08_native_run.log')
first = json.loads((ROOT/'native-run-1/reports/selection.json').read_text())
total = 576-len(json.loads((ROOT/'review_approval.json').read_text())['selected_bad'])
assert first['active']==total and not first['exact_duplicate_active']
published = json.loads((ROOT/'native-run-1/reports/published_release.json').read_text())
assert lab.command(['git','-C',REPO,'rev-parse','HEAD']).strip()==checkout
display(pd.DataFrame([dict(pulsar=p,**v) for p,v in first['pulsars'].items()]))

In [ ]:
# A fresh build uses the same config and decisions; only its output path changes.
lab.command(CLI+['run','--config',profile_path,'--set',
    'output.directory="../../native-run-2"'],log=ROOT/'logs/08_native_repeat.log')
second = json.loads((ROOT/'native-run-2/reports/selection.json').read_text())
assert first==second
assert lab.hashes(ROOT/'native-run-1/dataset')==lab.hashes(ROOT/'native-run-2/dataset')
(ROOT/'release_run.json').write_text(json.dumps(dict(commit=published['commit'],
    active=total,profile=str(profile_path)),indent=2))

## Test the locks

Make a copy of the config directory, alter a decision file, then run the copy.
It must fail before creating output. Do not update the expected hashes to make
an unexplained failure pass. Input changes require a reviewed new release profile.

In [ ]:
from pleb.release import ReleaseError, run_release
broken = CONFIGS/'tampered-native'
shutil.copytree(native,broken)
(broken/'decisions.json').write_text('{}')
try:
    run_release(broken/'release.toml',overrides=['output.directory="../../must-not-build"'])
except ReleaseError as error:
    assert 'checksum' in str(error)
    print('Expected rejection:',error)
else:
    raise AssertionError('Changed decision input was silently accepted')
assert not (ROOT/'must-not-build').exists()

## Native human review on a new release

Set `qc.mode='detect'` with explicit detector/application settings and
`review.mode='interactive'` in a new profile. `pleb run --config release.toml`
pauses after QC. Run the printed `pleb review --run-dir .../review` command,
record reasons/reviewer, then `pleb run --config release.toml --resume`.
Decisions are saved in `reports/review_decisions.json`; use that file as the
catalog for independent reconstruction. Never change a run's config midway.

**Your task:** transfer raw, config, catalogs and the pinned runtime to another
workspace. Rebuild there and compare source IDs, states and measurement hashes.
For a real PTA also review backend priorities, telescope clocks, events/eclipses,
fit settings and residual diagnostics for every published variant. A passed
selection test is not a claim that exclusions were scientifically justified.